# 02 · Everything Is a Token: Discretizing Four Modalities

**Hardware**: 🟢 CPU is fine (VAE weights ~300MB, EnCodec ~100MB)

The first principle of modern multimodal models: **turn any modality into a token sequence, and let a Transformer do the rest**. In this notebook you perform four conversions by hand:

| Modality | Method | Hands-on here |
|---|---|---|
| Text | BPE subwords | GPT-2 tokenizer |
| Image (understanding) | ViT patchify | hand-rolled patchify |
| Image (generation) | VAE latent space | SD VAE encode/decode |
| Audio (generation) | neural codec tokens | EnCodec |

Theory reference: [theory.md](../theory.md) §1.

In [ ]:
%pip install -q torch transformers diffusers datasets soundfile matplotlib pillow requests

In [ ]:
import torch
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"  # CPU is fast enough throughout
print(f"device = {device}")

## 1. Text → BPE tokens

BPE starts from characters and repeatedly merges frequent adjacent pairs into a subword vocabulary. Two things to observe:

1. Common English words are single tokens; rare words get shattered
2. GPT-2's vocabulary is hostile to Chinese — one character often costs 2–3 byte-level tokens (modern vocabularies like Qwen's are far more efficient)

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")

for text in [
    "Multimodal models turn everything into tokens.",
    "Antidisestablishmentarianism",
    "多模态模型把一切变成 token。",
]:
    ids = tok.encode(text)
    pieces = [tok.decode([i]) for i in ids]
    print(f"{text}\n  -> {len(ids)} tokens: {pieces}\n")

## 2. Image → ViT patches (understanding side)

ViT's approach is blunt: slice the image into 16×16 (or 14×14) squares, flatten each, pass it through a linear layer — and you have "visual words". A 224×224 image = **196 patch tokens** — the number that governs a VLM's image cost.

In [ ]:
import requests
from io import BytesIO
from PIL import Image
import numpy as np

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
img = Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB").resize((224, 224))
x = torch.tensor(np.array(img), dtype=torch.float32).permute(2, 0, 1) / 255.0  # [3,224,224]

P = 16
patches = x.unfold(1, P, P).unfold(2, P, P)          # [3, 14, 14, 16, 16]
patches = patches.permute(1, 2, 0, 3, 4)             # [14, 14, 3, 16, 16]
print(f"patch grid: {patches.shape[0]}x{patches.shape[1]} = {patches.shape[0]*patches.shape[1]} tokens")
print(f"each patch flattens to {3*P*P} dims (then a linear layer projects to model width)")

fig, axes = plt.subplots(14, 14, figsize=(6, 6))
for i in range(14):
    for j in range(14):
        axes[i, j].imshow(patches[i, j].permute(1, 2, 0))
        axes[i, j].axis("off")
plt.suptitle("One image = 196 patch tokens")
plt.show()

**Think about it**: a high-res document page (say 1536×1536) sliced at 16px is 9,216 tokens — which is exactly why the "dynamic resolution" and "visual token compression" techniques of chapters 01/02 exist.

## 3. Image → VAE latent (generation side)

Generators don't work in pixels — too expensive. Stable Diffusion's VAE squeezes a 512×512×3 image into a 64×64×4 latent (**48× compression**); the whole diffusion process happens in that latent space.

In [ ]:
from diffusers import AutoencoderKL

vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device).eval()

xb = (x.unsqueeze(0).to(device) * 2 - 1)  # VAE expects [-1, 1]
with torch.no_grad():
    latent = vae.encode(xb).latent_dist.sample()
    recon = vae.decode(latent).sample

print(f"pixels: {tuple(xb.shape)} = {xb.numel():,} values")
print(f"latent: {tuple(latent.shape)} = {latent.numel():,} values ({xb.numel()/latent.numel():.0f}x compression)")

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(img); axes[0].set_title("original"); axes[0].axis("off")
for c in range(4):
    axes[c+1].imshow(latent[0, c].cpu(), cmap="RdBu")
    axes[c+1].set_title(f"latent ch{c}"); axes[c+1].axis("off")
rec = ((recon[0].cpu().clamp(-1, 1) + 1) / 2).permute(1, 2, 0)
axes[5].imshow(rec); axes[5].set_title("VAE reconstruction"); axes[5].axis("off")
plt.show()

Notice the 4 latent channels faintly preserve spatial structure — this is not mystery compression but a "semantic thumbnail". The generation-side DiT runs flow matching on this 28×28×4 tensor (for a 224 input), patchifying it once more into DiT input tokens.

## 4. Audio → EnCodec discrete tokens

The cornerstone of audio generation: neural codecs use **residual vector quantization (RVQ)** to turn waveforms into multi-layer discrete tokens. The first codebook captures coarse structure; each further layer quantizes the residual — more layers, better fidelity. These are exactly the target sequences that chapter 06's TTS (the VALL-E paradigm) generates.

In [ ]:
from transformers import EncodecModel, AutoProcessor
from datasets import load_dataset, Audio

codec = EncodecModel.from_pretrained("facebook/encodec_24khz").to(device).eval()
codec_proc = AutoProcessor.from_pretrained("facebook/encodec_24khz")

ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
ds = ds.cast_column("audio", Audio(sampling_rate=24000))
wav = ds[0]["audio"]["array"]
print(f"audio length: {len(wav)/24000:.1f}s")

inputs = codec_proc(raw_audio=wav, sampling_rate=24000, return_tensors="pt").to(device)
with torch.no_grad():
    enc = codec.encode(inputs["input_values"], inputs["padding_mask"], bandwidth=6.0)

codes = enc.audio_codes  # [chunks, batch, n_q, frames]
print(f"codes shape: {tuple(codes.shape)}")
print(f"-> {codes.shape[2]} RVQ codebook layers x 75 frames/sec, codebook size 1024")
print(f"first 10 tokens (layer 0): {codes[0, 0, 0, :10].tolist()}")

In [ ]:
# Listen: reconstruction quality at different bandwidths (= number of RVQ layers)
from IPython.display import Audio as AudioPlayer, display

print("Original audio:")
display(AudioPlayer(wav, rate=24000))

for bw in [1.5, 6.0, 24.0]:
    with torch.no_grad():
        e = codec.encode(inputs["input_values"], inputs["padding_mask"], bandwidth=bw)
        d = codec.decode(e.audio_codes, e.audio_scales, inputs["padding_mask"])[0]
    n_q = e.audio_codes.shape[2]
    print(f"bandwidth={bw}kbps ({n_q} codebooks, {n_q*75} tokens/sec):")
    display(AudioPlayer(d[0].cpu().numpy(), rate=24000))

## Wrap-up: a token exchange-rate table

| Content | Tokens (order of magnitude) |
|---|---|
| One English sentence (10 words) | ~13 |
| One 224×224 image (ViT-16) | 196 |
| One 1536×1536 document page | ~9,000 (pre-compression) |
| One second of audio (EnCodec 6kbps) | 600 (8 layers × 75 frames) |
| One second of 480p video (spacetime patches) | ~10,000+ |

This table explains nearly every engineering decision later in the book: why VLMs compress visual tokens (ch. 01), why video generation burns the most money (ch. 05), and why "rendering long text as images" can *save* tokens (ch. 02's optical compression).

## Exercises

1. Swap the tokenizer for `Qwen/Qwen3-0.6B`'s and compare Chinese efficiency.
2. Latent arithmetic: interpolate the latents of two images and decode — what do you get?
3. Reconstruct with EnCodec's lowest bandwidth setting and listen: what information is lost — timbre or content?